# POC 1: The Pre-Call Brief

**Pain point:** You have a 30-min catch-up tomorrow with a stakeholder you haven't spoken to in 6 weeks. You spend 15 minutes hunting through emails to reconstruct context — or walk in cold.

**What this notebook shows:** Grounding a model with multi-source context (meeting notes + emails + project updates) produces a specific, actionable pre-call brief. Without grounding, the model gives generic advice that could apply to any meeting with anyone.

**You need:** A free Groq API key from https://console.groq.com

**Connecting Dots**, is where I write about the patterns I notice while building, checkout my blogs for more

👉 https://sriharshacr.github.io/blogs/

In [ ]:
!pip install groq -q

In [6]:
import os
from groq import Groq

In [8]:
MODEL = 'openai/gpt-oss-120b'  # or 'qwen/qwen3.8-27b'

In [7]:
# ⚠️ DO NOT PASTE any API key here
# ✅ Set the API key in Colab secret named GROQ_API_KEY
try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
except Exception:
    GROQ_API_KEY = os.environ.get('GROQ_API_KEY') or input('Enter Groq API key: ')

client = Groq(api_key=GROQ_API_KEY)

In [ ]:
def call_llm(system_prompt, user_message, temperature=0.3, max_tokens=800):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_message}
        ],
        temperature=temperature,
        max_tokens=max_tokens
    )
    return response.choices[0].message.content

print(f'Model ready: {MODEL}')

In [ ]:
# --- Synthetic grounding material ---
# Replace with your own meeting notes, emails, and project updates.

LAST_MEETING_NOTES = """
Meeting: Budget Reforecast Review | 14 Jul 2025 | Attendees: Harsha, Sarah (Finance Director)

- Sarah requested a cost centre rollup for Q3 — wants to see cloud, consulting, and headcount separated
- Harsha committed to sending the rollup by 31 July
- Discussed potential overspend on cloud infra; Sarah flagged it may affect Q4 headcount approvals
- Sarah mentioned the new finance system (Workday) is being rolled out in October — she will be the business owner
- No action required from Harsha on the Workday rollout yet
"""

EMAILS_SINCE = """
Email 1 — 4 Aug 2025, Sarah to Harsha:
  Subject: Re: Q3 cost centre rollup
  Body: Hi Harsha, just checking in on the cost centre rollup — it was due end of July.
  Can you give me an ETA? I need it before the ExCo pack on 20 August.

Email 2 — 5 Aug 2025, Harsha to Sarah:
  Subject: Re: Q3 cost centre rollup
  Body: Hi Sarah, apologies for the delay. The India team numbers came in late.
  We're at about 80% — I can send a draft by 12 Aug and final by 15 Aug. Does that work?

Email 3 — 5 Aug 2025, Sarah to Harsha:
  Body: 15 Aug works. Please flag if there are any surprises in the numbers beforehand.

Email 4 — 18 Aug 2025, Sarah (forward to team):
  Subject: FWD: Workday go-live update
  Body: FYI — Workday is now confirmed for 1 Oct. Finance teams will be in training w/c 15 Sep.
  All cost centre owners need to validate their chart of accounts by 5 Sep.
"""

PROJECT_STATUS = """
Project: Q3 Budget Rollup
Status: Final version sent to Sarah on 14 Aug (one day before deadline).
Key finding: Cloud infra overspend is £38k vs budget — driven by the unplanned data migration.
Sarah acknowledged receipt but has not yet responded with feedback.
Outstanding: chart of accounts validation for Workday due 5 Sep — not yet started.
"""

MEETING_CONTEXT = f"""
=== LAST MEETING NOTES (14 Jul) ===
{LAST_MEETING_NOTES}

=== EMAILS SINCE LAST MEETING ===
{EMAILS_SINCE}

=== CURRENT PROJECT STATUS ===
{PROJECT_STATUS}
"""

print('Grounding data loaded.')
print(f'Total context: ~{len(MEETING_CONTEXT.split())} words')

In [ ]:
# --- UNGROUNDED call ---
# The model knows nothing about Sarah, the history, or the current state.

ungrounded_system = "You are a helpful professional assistant."

ungrounded_query = """
I have a 30-minute catch-up call tomorrow with Sarah, our Finance Director.
We haven't spoken in about 6 weeks. Give me a quick context card to prepare.
"""

print('=== UNGROUNDED OUTPUT ===')
print(call_llm(ungrounded_system, ungrounded_query))

In [ ]:
# --- GROUNDED call ---
# Same question, but the model has the actual context.

grounded_system = """
You are a professional context synthesizer. Given notes from previous interactions,
emails, and project updates, produce a concise pre-call brief.

Output exactly five bullets:
1. What was agreed or committed to last time
2. What has changed or happened since
3. What the other person's likely concern or agenda is today
4. What you should proactively bring up
5. What NOT to re-open (resolved or irrelevant items)

Be specific. Use names, dates, and numbers from the provided material.
Do not add generic advice. Do not pad.
"""

grounded_query = f"""
I have a 30-minute catch-up with Sarah (Finance Director) tomorrow.
Here is the full context from our last interactions:

{MEETING_CONTEXT}

Give me my pre-call brief.
"""

print('=== GROUNDED OUTPUT ===')
print(call_llm(grounded_system, grounded_query))

## What just happened

The **ungrounded** output gives generic meeting-prep advice — applicable to any stakeholder conversation. It cannot mention the budget rollup, the cloud overspend, or the Workday chart-of-accounts deadline because it doesn't know they exist.

The **grounded** output names specific commitments, flags the unresolved item (Workday validation due 5 Sep), and tells you what Sarah's likely agenda is based on her last email.

**Token cost note:** The grounding context here is ~350 tokens. The savings come from avoiding a long clarification session at the start of the call — or from avoiding the follow-up email you'd need to send after walking in underprepared. This is Scenario 2 from the series: selective injection of only what's relevant.